# 05 · Clasificación Fake vs Real

Entrenamiento + comparación LogReg vs Naive Bayes sobre TF-IDF.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src import config
from src.data_loader import train_val_test_split
from src.classification import FakeNewsClassifier

df = pd.read_parquet(config.CORPUS_PATH)
train, val, test = train_val_test_split(df)
len(train), len(val), len(test)

In [ ]:
results = {}
for kind in ('logreg', 'nb'):
    clf = FakeNewsClassifier(kind=kind).fit(train['text'].tolist(), train['label'].tolist())
    metrics = clf.evaluate(test['text'].tolist(), test['label'].tolist())
    results[kind] = metrics
    print(f'\n=== {kind.upper()} ===')
    print(pd.DataFrame(metrics['report']).T.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, (kind, m) in zip(axes, results.items()):
    sns.heatmap(m['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['FAKE','REAL'], yticklabels=['FAKE','REAL'])
    ax.set_title(kind.upper())
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'confusion_matrices.png', dpi=120)
plt.show()

In [ ]:
# Persistir el mejor
best = FakeNewsClassifier(kind='logreg').fit(train['text'].tolist(), train['label'].tolist())
best.save(config.CLASSIFIER_PATH)
print('Guardado en', config.CLASSIFIER_PATH)